# NumPyro NUTS vs Pyro Variational Inference: BNN Posterioru Kararı Değiştiriyor mu?

Bu notebook aynı küçük Bayesçi sinir ağı ailesini iki farklı çıkarım yöntemiyle karşılaştırır:

1. **Pyro + SVI + AutoDiagonalNormal** — mean-field variational inference (VI)
2. **NumPyro + NUTS** — Hamiltonian Monte Carlo ailesinden No-U-Turn Sampler

Amaç yalnızca tahmin hatasını karşılaştırmak değildir. Endüstri mühendisliği açısından asıl soru:

> Posterior yaklaşım yöntemi, belirsizlik tahminini ve dolayısıyla optimizasyon kararını değiştiriyor mu?

Akış:

```text
aynı veri + aynı BNN mimarisi + aynı prior
            ↓
        iki inference
       ↙            ↘
 Pyro VI         NumPyro NUTS
       ↘            ↙
 posterior predictive
            ↓
RMSE / coverage / interval width
            ↓
stok-üretim kararı
            ↓
beklenen maliyet / CVaR / servis riski
```

**Önemli:** NUTS "ground truth posterior" değildir. Sonlu MCMC örnekleri, convergence ve model spesifikasyonu yine kontrol edilmelidir. Ancak küçük/orta modellerde VI için güçlü bir posterior benchmark'ı olabilir.


In [ ]:
# Gerekirse:
# %pip install torch pyro-ppl jax numpyro numpy pandas matplotlib

import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import pyro
import pyro.distributions as pydist
from pyro.infer import SVI, Trace_ELBO, Predictive as PyroPredictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as ndist
from numpyro.infer import MCMC, NUTS, Predictive as NumPyroPredictive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)
torch.set_default_dtype(torch.float32)


## 1. Sentetik talep problemi

Aynı veri üretim mekanizmasını iki inference yöntemi de görür.

Özellikler:

- promosyon,
- hafta sonu,
- sıcaklık,
- sezon,
- trend.

Gerçek uygulamada bu kısım ERP/MES/IoT verisiyle değiştirilir. Sentetik kurulumun avantajı, out-of-sample kararları **gerçek veri üretim mekanizmasına karşı** test edebilmemizdir.


In [ ]:
def true_demand_parameters(X_raw):
    # kolonlar: promotion, weekend, temperature, season, trend
    promotion = X_raw[..., 0]
    weekend = X_raw[..., 1]
    temperature = X_raw[..., 2]
    season = X_raw[..., 3]
    trend = X_raw[..., 4]

    mu = (
        105.0
        + 18.0 * promotion
        + 8.0 * weekend
        + 13.0 * season
        + 0.45 * temperature
        + 8.0 * trend
        + 5.0 * promotion * season
    )
    sigma = 7.0
    return mu, sigma

n = 260
day = np.arange(n)
promotion = np.random.binomial(1, 0.25, n).astype(np.float32)
weekend = ((day % 7) >= 5).astype(np.float32)
temperature = (
    22 + 9*np.sin(2*np.pi*day/90) + np.random.normal(0, 2.0, n)
).astype(np.float32)
season = np.sin(2*np.pi*day/30).astype(np.float32)
trend = (day/(n-1)).astype(np.float32)

X_raw = np.column_stack(
    [promotion, weekend, temperature, season, trend]
).astype(np.float32)

true_mu, true_sigma = true_demand_parameters(X_raw)
y_raw = (true_mu + np.random.normal(0, true_sigma, n)).astype(np.float32)

# Zamansal holdout: son 60 gözlem test
n_train = 200
X_train_raw, X_test_raw = X_raw[:n_train], X_raw[n_train:]
y_train_raw, y_test_raw = y_raw[:n_train], y_raw[n_train:]

X_mean = X_train_raw.mean(axis=0, keepdims=True)
X_std = X_train_raw.std(axis=0, keepdims=True) + 1e-6
y_mean = float(y_train_raw.mean())
y_std = float(y_train_raw.std() + 1e-6)

X_train_np = ((X_train_raw - X_mean)/X_std).astype(np.float32)
X_test_np = ((X_test_raw - X_mean)/X_std).astype(np.float32)
y_train_np = ((y_train_raw - y_mean)/y_std).astype(np.float32)

print("Train:", X_train_np.shape, "Test:", X_test_np.shape)


## 2. Ortak BNN spesifikasyonu

Her iki modelde de:

\[
h=\tanh(XW_1+b_1),
\]

\[
\mu=h^\top w_2+b_2,
\]

ve

\[
Y\mid X,w,\sigma \sim \mathcal N(\mu,\sigma^2).
\]

Priorlar da aynı seçilir. Böylece karşılaştırılan temel unsur model ailesinden çok **posterior çıkarım yöntemidir**.

Gizli katman küçük tutulur; çünkü NUTS bütün ağırlık uzayında HMC yapar ve ağ büyüdükçe maliyet hızla artar.


In [ ]:
HIDDEN = 6
IN_DIM = X_train_np.shape[1]

class PyroBNN(PyroModule):
    def __init__(self):
        super().__init__()
        self.hidden = PyroModule[nn.Linear](IN_DIM, HIDDEN)
        self.hidden.weight = PyroSample(
            pydist.Normal(0, 0.8).expand([HIDDEN, IN_DIM]).to_event(2)
        )
        self.hidden.bias = PyroSample(
            pydist.Normal(0, 0.8).expand([HIDDEN]).to_event(1)
        )
        self.out = PyroModule[nn.Linear](HIDDEN, 1)
        self.out.weight = PyroSample(
            pydist.Normal(0, 0.8).expand([1, HIDDEN]).to_event(2)
        )
        self.out.bias = PyroSample(
            pydist.Normal(0, 0.8).expand([1]).to_event(1)
        )

    def forward(self, X, y=None):
        h = torch.tanh(self.hidden(X))
        mu = self.out(h).squeeze(-1)
        sigma = pyro.sample("sigma", pydist.LogNormal(-1.0, 0.35))
        pyro.deterministic("mu", mu)
        with pyro.plate("data", X.shape[0]):
            pyro.sample("obs", pydist.Normal(mu, sigma), obs=y)
        return mu


def numpyro_bnn(X, y=None):
    W1 = numpyro.sample(
        "W1", ndist.Normal(0, 0.8).expand((HIDDEN, IN_DIM)).to_event(2)
    )
    b1 = numpyro.sample(
        "b1", ndist.Normal(0, 0.8).expand((HIDDEN,)).to_event(1)
    )
    W2 = numpyro.sample(
        "W2", ndist.Normal(0, 0.8).expand((1, HIDDEN)).to_event(2)
    )
    b2 = numpyro.sample(
        "b2", ndist.Normal(0, 0.8).expand((1,)).to_event(1)
    )
    sigma = numpyro.sample("sigma", ndist.LogNormal(-1.0, 0.35))

    h = jnp.tanh(X @ W1.T + b1)
    mu = (h @ W2.T).squeeze(-1) + b2.squeeze(-1)
    numpyro.deterministic("mu", mu)

    with numpyro.plate("data", X.shape[0]):
        numpyro.sample("obs", ndist.Normal(mu, sigma), obs=y)

    return mu


## 3. Pyro: mean-field Variational Inference

`AutoDiagonalNormal`, latent değişkenlerin dönüştürülmüş uzayında diagonal Gaussian posterior kullanır.

Avantaj:

- hızlı,
- büyük modellere NUTS'tan daha kolay ölçeklenir.

Sınırlama:

- posterior korelasyonlarını doğrudan temsil etmez,
- multimodality ve karmaşık geometriyi kaçırabilir,
- bazı problemlerde posterior varyansını düşük tahmin edebilir.

Son madde otomatik bir sonuç değildir; bu notebook bunu ampirik olarak test eder.


In [ ]:
X_train_t = torch.tensor(X_train_np)
X_test_t = torch.tensor(X_test_np)
y_train_t = torch.tensor(y_train_np)

pyro.clear_param_store()
pyro_model = PyroBNN()
guide = AutoDiagonalNormal(pyro_model)
svi = SVI(
    pyro_model,
    guide,
    pyro.optim.Adam({"lr": 0.01}),
    loss=Trace_ELBO(),
)

vi_start = time.perf_counter()
vi_losses = []
for step in range(3000):
    loss = svi.step(X_train_t, y_train_t) / len(y_train_t)
    vi_losses.append(loss)
    if (step + 1) % 600 == 0:
        print(f"VI adım {step+1}: ELBO/gözlem={loss:.4f}")
vi_seconds = time.perf_counter() - vi_start

plt.figure(figsize=(8, 3))
plt.plot(vi_losses)
plt.xlabel("SVI adımı")
plt.ylabel("ELBO / gözlem")
plt.show()

print("VI eğitim süresi (sn):", round(vi_seconds, 2))


In [ ]:
pyro_test_predictive = PyroPredictive(
    pyro_model,
    guide=guide,
    num_samples=1500,
    return_sites=("obs", "mu"),
)
pyro_test = pyro_test_predictive(X_test_t)
vi_test_obs = (
    pyro_test["obs"].detach().cpu().numpy() * y_std + y_mean
)
vi_test_mu = (
    pyro_test["mu"].detach().cpu().numpy() * y_std + y_mean
)

print("VI posterior predictive shape:", vi_test_obs.shape)


## 4. NumPyro: NUTS

NUTS, Hamiltonian Monte Carlo'nun trajectory length'i otomatik seçen bir varyantıdır.

Burada küçük BNN üzerinde posterioru örnekleriz. MCMC için en az şu kontroller yapılmalıdır:

- divergence sayısı,
- effective sample size,
- \(\hat R\),
- yeterli warmup ve chain.

Öğretim süresini makul tutmak için tek chain kullanıyoruz. Ciddi bir analizde birden fazla chain ile \(\hat R\) kontrolü yapılmalıdır.


In [ ]:
X_train_j = jnp.asarray(X_train_np)
X_test_j = jnp.asarray(X_test_np)
y_train_j = jnp.asarray(y_train_np)

nuts_kernel = NUTS(
    numpyro_bnn,
    target_accept_prob=0.85,
    max_tree_depth=8,
)
mcmc = MCMC(
    nuts_kernel,
    num_warmup=700,
    num_samples=1000,
    num_chains=1,
    progress_bar=True,
)

nuts_start = time.perf_counter()
mcmc.run(
    jax.random.key(SEED),
    X_train_j,
    y_train_j,
    extra_fields=("diverging",),
)
nuts_seconds = time.perf_counter() - nuts_start

mcmc.print_summary()
extra = mcmc.get_extra_fields()
print("Divergence sayısı:", int(np.asarray(extra["diverging"]).sum()))
print("NUTS toplam süre (sn):", round(nuts_seconds, 2))


In [ ]:
nuts_samples = mcmc.get_samples()

nuts_predictive = NumPyroPredictive(
    numpyro_bnn,
    posterior_samples=nuts_samples,
    return_sites=["obs", "mu"],
)
nuts_test = nuts_predictive(
    jax.random.key(SEED + 1),
    X_test_j,
    None,
)

nuts_test_obs = np.asarray(nuts_test["obs"]) * y_std + y_mean
nuts_test_mu = np.asarray(nuts_test["mu"]) * y_std + y_mean

print("NUTS posterior predictive shape:", nuts_test_obs.shape)


## 5. Predictive kalite: RMSE, coverage ve interval width

Point accuracy tek başına belirsizlik kalitesini göstermez.

90% posterior predictive interval için:

- **coverage** yaklaşık 0.90 olmalı,
- interval gereksiz geniş de olmamalı.

Bu nedenle RMSE + coverage + interval width birlikte raporlanır.


In [ ]:
def predictive_metrics(samples, y_true, interval=0.90):
    mean_pred = samples.mean(axis=0)
    alpha = 1 - interval
    lo = np.quantile(samples, alpha/2, axis=0)
    hi = np.quantile(samples, 1-alpha/2, axis=0)

    rmse = np.sqrt(np.mean((mean_pred - y_true)**2))
    coverage = np.mean((y_true >= lo) & (y_true <= hi))
    width = np.mean(hi - lo)
    return rmse, coverage, width, mean_pred, lo, hi

vi_rmse, vi_cov, vi_width, vi_mean, vi_lo, vi_hi = predictive_metrics(
    vi_test_obs, y_test_raw
)
nuts_rmse, nuts_cov, nuts_width, nuts_mean, nuts_lo, nuts_hi = predictive_metrics(
    nuts_test_obs, y_test_raw
)

metrics = pd.DataFrame({
    "Inference": ["Pyro VI", "NumPyro NUTS"],
    "RMSE": [vi_rmse, nuts_rmse],
    "90% coverage": [vi_cov, nuts_cov],
    "Ortalama interval genişliği": [vi_width, nuts_width],
    "Yaklaşık süre (sn)": [vi_seconds, nuts_seconds],
})
metrics


In [ ]:
idx = np.arange(len(y_test_raw))

plt.figure(figsize=(11, 4))
plt.plot(idx, y_test_raw, "o", ms=3, label="Gerçek")
plt.plot(idx, vi_mean, label="VI ortalama")
plt.fill_between(idx, vi_lo, vi_hi, alpha=0.2, label="VI %90 PI")
plt.xlabel("Test gözlemi")
plt.ylabel("Talep")
plt.legend()
plt.show()

plt.figure(figsize=(11, 4))
plt.plot(idx, y_test_raw, "o", ms=3, label="Gerçek")
plt.plot(idx, nuts_mean, label="NUTS ortalama")
plt.fill_between(idx, nuts_lo, nuts_hi, alpha=0.2, label="NUTS %90 PI")
plt.xlabel("Test gözlemi")
plt.ylabel("Talep")
plt.legend()
plt.show()


## 6. Aynı gelecek koşulu için posterior predictive

Şimdi inference farkını doğrudan bir karar girdisine çeviriyoruz.

Gelecek durum:

- promosyon var,
- hafta sonu,
- sıcaklık yüksek,
- sezon pozitif,
- trend eğitim aralığının biraz ötesinde.

Bu nokta epistemik belirsizliğin daha görünür olabileceği bir bölgedir.


In [ ]:
future_raw = np.array([[1.0, 1.0, 34.0, 0.75, 1.08]], dtype=np.float32)
future_scaled = ((future_raw - X_mean)/X_std).astype(np.float32)

future_t = torch.tensor(future_scaled)
vi_future = PyroPredictive(
    pyro_model,
    guide=guide,
    num_samples=4000,
    return_sites=("obs", "mu"),
)(future_t)

vi_future_obs = (
    vi_future["obs"].detach().cpu().numpy().reshape(-1) * y_std + y_mean
)
vi_future_mu = (
    vi_future["mu"].detach().cpu().numpy().reshape(-1) * y_std + y_mean
)

nuts_future = NumPyroPredictive(
    numpyro_bnn,
    posterior_samples=nuts_samples,
    return_sites=["obs", "mu"],
)(
    jax.random.key(SEED + 2),
    jnp.asarray(future_scaled),
    None,
)

nuts_future_obs = np.asarray(nuts_future["obs"]).reshape(-1) * y_std + y_mean
nuts_future_mu = np.asarray(nuts_future["mu"]).reshape(-1) * y_std + y_mean

future_summary = pd.DataFrame({
    "Inference": ["Pyro VI", "NumPyro NUTS"],
    "Predictive mean": [vi_future_obs.mean(), nuts_future_obs.mean()],
    "Predictive std": [vi_future_obs.std(ddof=1), nuts_future_obs.std(ddof=1)],
    "Epistemik mean std": [vi_future_mu.std(ddof=1), nuts_future_mu.std(ddof=1)],
    "P90": [np.quantile(vi_future_obs, .90), np.quantile(nuts_future_obs, .90)],
    "P95": [np.quantile(vi_future_obs, .95), np.quantile(nuts_future_obs, .95)],
})
future_summary


## 7. Inference yöntemi optimizasyon kararını değiştiriyor mu?

Tek dönemlik üretim/stok kararında:

\[
L(q,D)=c_h(q-D)^+ + c_u(D-q)^+.
\]

İki karar kriteri karşılaştıracağız:

1. **Beklenen maliyet minimizasyonu**
2. **CVaR\(_{0.95}\) minimizasyonu**

Aynı maliyet fonksiyonu kullanılıyor; tek fark posterior predictive talep senaryolarının VI veya NUTS'tan gelmesi.


In [ ]:
holding_cost = 1.0
shortage_cost = 7.0
capacity = 190.0
q_grid = np.linspace(70, capacity, 1201)

def losses_for_q(q, demands):
    return (
        holding_cost * np.maximum(q - demands, 0)
        + shortage_cost * np.maximum(demands - q, 0)
    )

def empirical_cvar(costs, alpha=0.95):
    eta = np.quantile(costs, alpha)
    return eta + np.maximum(costs - eta, 0).mean() / (1-alpha)

def optimize_from_samples(demands):
    exp_cost = np.array([losses_for_q(q, demands).mean() for q in q_grid])
    cvar95 = np.array([
        empirical_cvar(losses_for_q(q, demands), 0.95)
        for q in q_grid
    ])
    return (
        q_grid[np.argmin(exp_cost)],
        q_grid[np.argmin(cvar95)],
        exp_cost.min(),
        cvar95.min(),
    )

vi_decision = optimize_from_samples(vi_future_obs)
nuts_decision = optimize_from_samples(nuts_future_obs)

decision_table = pd.DataFrame({
    "Inference": ["Pyro VI", "NumPyro NUTS"],
    "q - beklenen maliyet": [vi_decision[0], nuts_decision[0]],
    "q - CVaR95": [vi_decision[1], nuts_decision[1]],
    "Posterior min beklenen maliyet": [vi_decision[2], nuts_decision[2]],
    "Posterior min CVaR95": [vi_decision[3], nuts_decision[3]],
})
decision_table


## 8. Sentetik gerçek dağılım altında out-of-sample karar testi

Gerçek uygulamada gerçek dağılım bilinmez; bağımsız holdout / ileri dönem gözlemleri kullanılır.

Sentetik örnekte veri üretim mekanizmasını bildiğimiz için iki inference yönteminin önerdiği kararları aynı "gerçek" gelecek dağılım altında test edebiliriz.


In [ ]:
future_true_mu, future_true_sigma = true_demand_parameters(future_raw)
rng = np.random.default_rng(SEED + 99)
true_future_demands = rng.normal(
    float(future_true_mu[0]),
    float(future_true_sigma),
    100_000,
)

def evaluate_decision(q, demands):
    costs = losses_for_q(q, demands)
    return {
        "q": q,
        "Beklenen gerçek maliyet": costs.mean(),
        "Gerçek CVaR95": empirical_cvar(costs, 0.95),
        "Stockout olasılığı": np.mean(demands > q),
    }

rows = []
for method, dec in [("Pyro VI", vi_decision), ("NumPyro NUTS", nuts_decision)]:
    for objective, q in [
        ("Beklenen maliyet", dec[0]),
        ("CVaR95", dec[1]),
    ]:
        out = evaluate_decision(q, true_future_demands)
        rows.append({"Inference": method, "Karar kriteri": objective, **out})

out_of_sample = pd.DataFrame(rows)
out_of_sample


## 9. Ne sonuç çıkarmalıyız?

Bu notebookta amaç NUTS'ı otomatik kazanan ilan etmek değildir.

Olası sonuçlar:

### VI ve NUTS benzer karar veriyorsa

Bu iyi bir pratik sonuçtur. Daha ucuz VI, downstream karar için yeterli olabilir.

### Posterior ortalamaları benzer ama interval / CVaR kararları ayrışıyorsa

Tahmin ortalaması yeterli değildir. Posterior geometrisi ve uncertainty calibration karar kalitesini etkiliyor demektir.

### NUTS diagnostics kötüyse

NUTS sonuçlarını referans kabul etmeyin. Divergence, düşük effective sample size veya chain uyumsuzluğu modelin / parametrizasyonun iyileştirilmesini gerektirir.

### VI daha dar posterior veriyorsa

Bu mean-field VI'ın bilinen olası davranışlarından biridir; fakat her veri/modelde olmak zorunda değildir. Ampirik coverage ve downstream risk testiyle doğrulanmalıdır.

---

## Endüstri mühendisliği açısından öneri

Bir BNN + optimizasyon çalışmasında inference seçimini şu şekilde değerlendirmek daha anlamlıdır:

\[
\text{inference}
\rightarrow
\text{posterior calibration}
\rightarrow
\text{scenario distribution}
\rightarrow
\text{decision}
\rightarrow
\text{out-of-sample cost/risk}.
\]

Sadece ELBO veya MCMC log-density kıyaslamak, karar sistemi için yeterli değildir.

### Daha ileri çalışmalar

- NUTS için 4 chain ve \(\hat R\) analizi
- NumPyro SVI vs NUTS
- Pyro `AutoLowRankMultivariateNormal` vs `AutoDiagonalNormal`
- Deep Ensemble ve Gaussian Process baseline
- heteroskedastik likelihood
- posterior predictive calibration
- aynı karşılaştırmayı CVaR / chance-constrained MILP üzerinde yapmak
